# L'anatomie de M2 · *The anatomy of M2*

Notebook compagnon du chapitre **25. Masse monétaire M1, M2 : ce que ces agrégats mesurent vraiment** — [lire l'article](https://nmlab.io/ressources/masse-monetaire-m1-m2).
Companion notebook to chapter **25. Money Supply M1, M2: What These Aggregates Really Measure** — [read the article](https://nmlab.io/en/ressources/money-supply-m1-m2).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    fig = nm.figure(1010); ax = nm.blank_axes(fig)
    d = dict(fr=("L'anatomie de M2 américain","Composantes de M2, mai 2026, en milliards de dollars — le numéraire n'en fait qu'un dixième.",
                 [("Numéraire","2 370",C["amber"]),("Dépôts à vue","6 976",C["blue"]),
                  ("Autres dépôts liquides","10 407",C["blue2"]),
                  ("Dépôts à terme < 100 000 $","1 026",C["teal"]),("Fonds monétaires de détail","2 275",C["green"])],
                 "M1 = 19 751","M2 = 23 052","Les « autres dépôts liquides » abritent l'épargne reclassée en 2020. Source : FRED (M2SL et composantes)."),
             en=("The anatomy of U.S. M2","Components of M2, May 2026, in billions of dollars — currency is only a tenth of it.",
                 [("Currency","2,370",C["amber"]),("Demand deposits","6,976",C["blue"]),
                  ("Other liquid deposits","10,407",C["blue2"]),
                  ("Small time deposits < $100k","1,026",C["teal"]),("Retail money funds","2,275",C["green"])],
                 "M1 = 19,751","M2 = 23,052","\"Other liquid deposits\" hold the savings reclassified in 2020. Source: FRED (M2SL and components)."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    comps=t[2]; vals=[2370,6976,10407,1026,2275]; total=sum(vals)
    x0=90; xw=W-180; ybar=470; hbar=175
    xc=x0
    for (lab,amt,col),v in zip(comps,vals):
        w=xw*v/total
        ax.add_patch(plt.Rectangle((xc,ybar),w,hbar,facecolor=col,edgecolor=C["bg"],lw=2.5,zorder=3))
        if w>110:
            ax.text(xc+w/2,ybar+hbar/2,amt,ha="center",va="center",fontsize=18.5,color=C["bg"],fontweight="bold",zorder=4)
        else:
            ax.text(xc+w/2,ybar+hbar+20,amt,ha="center",va="bottom",fontsize=15.5,color=C["text"],fontweight="bold")
        xc+=w
    # total M2 au-dessus, à gauche
    ax.text(x0, ybar+hbar+118, t[4], ha="left", va="center", fontsize=21, color=C["teal"], fontweight="bold")
    # accolade M1 sous la barre (3 premiers segments)
    m1w=xw*sum(vals[:3])/total
    yb=ybar-38
    ax.plot([x0,x0+m1w],[yb,yb],color=C["blue"],lw=2.4)
    ax.plot([x0,x0],[yb,yb+10],color=C["blue"],lw=2.4)
    ax.plot([x0+m1w,x0+m1w],[yb,yb+10],color=C["blue"],lw=2.4)
    ax.text(x0+m1w/2, yb-34, t[3], ha="center", va="center", fontsize=20, color=C["blue"], fontweight="bold")
    # légende deux lignes (colonnes fixes) sous l'accolade
    def item(x,y,lab,col):
        ax.add_patch(plt.Rectangle((x,y-15),30,30,facecolor=col,edgecolor="none",zorder=3))
        ax.text(x+46,y,lab,ha="left",va="center",fontsize=16.5,color=C["text"])
    row1=comps[:3]; row2=comps[3:]
    for (lab,amt,col),x in zip(row1,[90,610,1130]): item(x,320,lab,col)
    for (lab,amt,col),x in zip(row2,[90,820]): item(x,232,lab,col)
    nm.footer(fig,t[5]);
    return fig


build_figure(LANG)